# Agent的高级用法-流式输出

## 1、values输出模式

当`stream_mode`设置为values模式时，每个步骤执行后，都会输出完整的状态信息，适用于每一步都要获取完整状态、状态持久化场景。

In [1]:
import truststore
from dataclasses import dataclass

from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 从.env文件中加载环境变量
load_dotenv(override=True)
truststore.inject_into_ssl()

# CLOSEAI_API_KEY = os.getenv("CLOSEAI_API_KEY")
# CLOSEAI_BASE_URL = os.getenv("CLOSEAI_BASE_URL")
#
# model = init_chat_model(
#     model="gpt-5.4-mini",
#     model_provider="openai",
#     api_key=CLOSEAI_API_KEY,
#     base_url=CLOSEAI_BASE_URL
# )
model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    extra_body={"thinking": {"type": "disabled"}},
)

In [2]:
from langchain.agents import create_agent
from langchain.tools import tool
from typing import Dict, Any
from rich import print as rprint

@tool
def query_customer_data(customer_id: str) -> Dict[str, Any]:
    """
    查询客户基本信息

    Args:
        customer_id: 客户ID，用于唯一标识客户

    Returns:
        包含客户基本信息的字典，如姓名、等级、加入日期等
    """
    # 模拟数据库查询
    return {"name": "张三","level": "VIP","join_date": "2023-01-15"}


@tool
def check_order_history(customer_id: str) -> Dict[str, Any]:
    """
    查询客户订单历史

    Args:
        customer_id: 客户ID，用于唯一标识客户

    Returns:
        包含客户订单历史的字典，如总订单数、总花费等
    """
    return {"total_orders": 15,"total_spent": 25800.00}


@tool
def get_current_promotions() -> Dict[str, Any]:
    """
    获取当前可用促销活动

    Returns:
        包含当前可用促销活动的字典，如活动名称、有效日期等
    """
    return {
        "promotions": ["老用户优惠", "会员专属折扣"],
        "valid_until": "2027-01-31"
    }


# 创建客户服务Agent
customer_service_agent = create_agent(
    model=model,
    tools=[query_customer_data, check_order_history, get_current_promotions]
)

for chunk in customer_service_agent.stream(
    {
        "messages" : [
            {"role":"user","content":"查询客户id为cust1234的完整的信息、历史订单和可用优惠"}
        ]
    },
    stream_mode="values"
):
    rprint(chunk)
    print("-" * 50)


{
    'messages': [
        HumanMessage(
            content='查询客户id为cust1234的完整的信息、历史订单和可用优惠',
            additional_kwargs={},
            response_metadata={},
            id='364e46c0-e37a-4cf9-a31e-7fde6da10ffd'
        )
    ]
}

--------------------------------------------------


{
    'messages': [
        HumanMessage(
            content='查询客户id为cust1234的完整的信息、历史订单和可用优惠',
            additional_kwargs={},
            response_metadata={},
            id='364e46c0-e37a-4cf9-a31e-7fde6da10ffd'
        ),
        AIMessage(
            content='好的，我来查询客户ID为cust1234的完整信息、历史订单和当前可用优惠。',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 120,
                    'prompt_tokens': 455,
                    'total_tokens': 575,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
                    'prompt_cache_hit_tokens': 0,
                    'prompt_cache_miss_tokens': 455
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402',
                'id': 'b5d89c3d-56bd-4694-87b9-3025166c63ca',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f7522-d5c9-7282-b45e-7a48078065d3-0',
            tool_calls=[
                {
                    'name': 'query_customer_data',
                    'args': {'customer_id': 'cust1234'},
                    'id': 'call_00_eRG05Dutz7Pema8VTNqY9470',
                    'type': 'tool_call'
                },
                {
                    'name': 'check_order_history',
                    'args': {'customer_id': 'cust1234'},
                    'id': 'call_01_2HKq8imL1e8jcjueTudk6027',
                    'type': 'tool_call'
                },
                {
                    'name': 'get_current_promotions',
                    'args': {},
                    'id': 'call_02_Bk8khJmab9EHLu1HglVS6784',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 455,
                'output_tokens': 120,
                'total_tokens': 575,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
            }
        )
    ]
}

--------------------------------------------------


{
    'messages': [
        HumanMessage(
            content='查询客户id为cust1234的完整的信息、历史订单和可用优惠',
            additional_kwargs={},
            response_metadata={},
            id='364e46c0-e37a-4cf9-a31e-7fde6da10ffd'
        ),
        AIMessage(
            content='好的，我来查询客户ID为cust1234的完整信息、历史订单和当前可用优惠。',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 120,
                    'prompt_tokens': 455,
                    'total_tokens': 575,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
                    'prompt_cache_hit_tokens': 0,
                    'prompt_cache_miss_tokens': 455
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402',
                'id': 'b5d89c3d-56bd-4694-87b9-3025166c63ca',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f7522-d5c9-7282-b45e-7a48078065d3-0',
            tool_calls=[
                {
                    'name': 'query_customer_data',
                    'args': {'customer_id': 'cust1234'},
                    'id': 'call_00_eRG05Dutz7Pema8VTNqY9470',
                    'type': 'tool_call'
                },
                {
                    'name': 'check_order_history',
                    'args': {'customer_id': 'cust1234'},
                    'id': 'call_01_2HKq8imL1e8jcjueTudk6027',
                    'type': 'tool_call'
                },
                {
                    'name': 'get_current_promotions',
                    'args': {},
                    'id': 'call_02_Bk8khJmab9EHLu1HglVS6784',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 455,
                'output_tokens': 120,
                'total_tokens': 575,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content='{"name": "张三", "level": "VIP", "join_date": "2023-01-15"}',
            name='query_customer_data',
            id='1851127e-43df-48e2-8456-bf0bf2e3434c',
            tool_call_id='call_00_eRG05Dutz7Pema8VTNqY9470'
        ),
        ToolMessage(
            content='{"total_orders": 15, "total_spent": 25800.0}',
            name='check_order_history',
            id='9e1f4358-0120-4c5c-a9d8-806bb4e9ce1a',
            tool_call_id='call_01_2HKq8imL1e8jcjueTudk6027'
        ),
        ToolMessage(
            content='{"promotions": ["老用户优惠", "会员专属折扣"], "valid_until": "2027-01-31"}',
            name='get_current_promotions',
            id='f22149c5-3510-4226-9b7a-876e262855e3',
            tool_call_id='call_02_Bk8khJmab9EHLu1HglVS6784'
        )
    ]
}

--------------------------------------------------


{
    'messages': [
        HumanMessage(
            content='查询客户id为cust1234的完整的信息、历史订单和可用优惠',
            additional_kwargs={},
            response_metadata={},
            id='364e46c0-e37a-4cf9-a31e-7fde6da10ffd'
        ),
        AIMessage(
            content='好的，我来查询客户ID为cust1234的完整信息、历史订单和当前可用优惠。',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 120,
                    'prompt_tokens': 455,
                    'total_tokens': 575,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
                    'prompt_cache_hit_tokens': 0,
                    'prompt_cache_miss_tokens': 455
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402',
                'id': 'b5d89c3d-56bd-4694-87b9-3025166c63ca',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f7522-d5c9-7282-b45e-7a48078065d3-0',
            tool_calls=[
                {
                    'name': 'query_customer_data',
                    'args': {'customer_id': 'cust1234'},
                    'id': 'call_00_eRG05Dutz7Pema8VTNqY9470',
                    'type': 'tool_call'
                },
                {
                    'name': 'check_order_history',
                    'args': {'customer_id': 'cust1234'},
                    'id': 'call_01_2HKq8imL1e8jcjueTudk6027',
                    'type': 'tool_call'
                },
                {
                    'name': 'get_current_promotions',
                    'args': {},
                    'id': 'call_02_Bk8khJmab9EHLu1HglVS6784',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 455,
                'output_tokens': 120,
                'total_tokens': 575,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content='{"name": "张三", "level": "VIP", "join_date": "2023-01-15"}',
            name='query_customer_data',
            id='1851127e-43df-48e2-8456-bf0bf2e3434c',
            tool_call_id='call_00_eRG05Dutz7Pema8VTNqY9470'
        ),
        ToolMessage(
            content='{"total_orders": 15, "total_spent": 25800.0}',
            name='check_order_history',
            id='9e1f4358-0120-4c5c-a9d8-806bb4e9ce1a',
            tool_call_id='call_01_2HKq8imL1e8jcjueTudk6027'
        ),
        ToolMessage(
            content='{"promotions": ["老用户优惠", "会员专属折扣"], "valid_until": "2027-01-31"}',
            name='get_current_promotions',
            id='f22149c5-3510-4226-9b7a-876e262855e3',
            tool_call_id='call_02_Bk8khJmab9EHLu1HglVS6784'
        ),
        AIMessage(
            content='以下是客户 **cust1234** 的完整信息汇总：\n\n---\n\n### 🧑 客户基本信息\n| 项目 | 内容 
|\n|------|------|\n| **客户姓名** | 张三 |\n| **客户等级** | 🏆 VIP |\n| **加入日期** | 2023-01-15 |\n\n---\n\n###
📦 订单历史\n| 项目 | 数据 |\n|------|------|\n| **总订单数** | 15 笔 |\n| **总消费金额** | ¥25,800.00 
|\n\n---\n\n### 🎉 当前可用优惠活动\n| 优惠活动名称 | 有效期至 |\n|-------------|---------|\n| ✅ 老用户优惠 | 
2027-01-31 |\n| ✅ 会员专属折扣 | 2027-01-31 |\n\n---\n\n**总结：** 
客户张三为VIP客户，自2023年1月加入以来累计消费了15笔订单，总金额达25,800元。目前有**老用户优惠**和**会员专属折扣**
两项优惠可以使用，有效期均至2027年1月31日。',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 233,
                    'prompt_tokens': 671,
                    'total_tokens': 904,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 

--------------------------------------------------


## 2、updates输出模式

这种模式就是默认模式。该模式中，每个步骤执行后，只增量更新状态中发生变化的内容，用于监控Agent 执行进度，例如观察Agent决定调用工具、工具执行结果等步骤。

In [3]:
for chunk in customer_service_agent.stream(
    {
        "messages" : [
            {"role":"user","content":"查询客户id为cust1234的完整的信息、历史订单和可用优惠"}
        ]
    },
    stream_mode="updates"
):
    rprint(chunk)
    print("-" * 50)

{
    'model': {
        'messages': [
            AIMessage(
                content='我来为您查询客户 cust1234 的完整信息、历史订单和当前可用优惠。',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 118,
                        'prompt_tokens': 455,
                        'total_tokens': 573,
                        'completion_tokens_details': None,
                        'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 384},
                        'prompt_cache_hit_tokens': 384,
                        'prompt_cache_miss_tokens': 71
                    },
                    'model_provider': 'deepseek',
                    'model_name': 'deepseek-v4-flash',
                    'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402',
                    'id': 'e1c1e655-2d02-4d26-a335-39b8697a9a37',
                    'finish_reason': 'tool_calls',
                    'logprobs': None
                },
                id='lc_run--019f7523-2f8a-7e20-819b-ce4f5cc8ccb4-0',
                tool_calls=[
                    {
                        'name': 'query_customer_data',
                        'args': {'customer_id': 'cust1234'},
                        'id': 'call_00_7UVvSEJv6kbf8FPT5yio1595',
                        'type': 'tool_call'
                    },
                    {
                        'name': 'check_order_history',
                        'args': {'customer_id': 'cust1234'},
                        'id': 'call_01_oc4fKYvCXtyNKjUCDmgV2903',
                        'type': 'tool_call'
                    },
                    {
                        'name': 'get_current_promotions',
                        'args': {},
                        'id': 'call_02_vcg4q1xvXOp6zRqqKbTq3206',
                        'type': 'tool_call'
                    }
                ],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 455,
                    'output_tokens': 118,
                    'total_tokens': 573,
                    'input_token_details': {'cache_read': 384},
                    'output_token_details': {}
                }
            )
        ]
    }
}

--------------------------------------------------


{
    'tools': {
        'messages': [
            ToolMessage(
                content='{"name": "张三", "level": "VIP", "join_date": "2023-01-15"}',
                name='query_customer_data',
                id='d4c30eca-f0c8-421b-b514-e39d5390a026',
                tool_call_id='call_00_7UVvSEJv6kbf8FPT5yio1595'
            )
        ]
    }
}

--------------------------------------------------


{
    'tools': {
        'messages': [
            ToolMessage(
                content='{"total_orders": 15, "total_spent": 25800.0}',
                name='check_order_history',
                id='43b0cefb-f300-4a7d-97c9-d45329e4fd61',
                tool_call_id='call_01_oc4fKYvCXtyNKjUCDmgV2903'
            )
        ]
    }
}

--------------------------------------------------


{
    'tools': {
        'messages': [
            ToolMessage(
                content='{"promotions": ["老用户优惠", "会员专属折扣"], "valid_until": "2027-01-31"}',
                name='get_current_promotions',
                id='1208bd2a-c181-4fb5-90c2-4bc8f07eba4b',
                tool_call_id='call_02_vcg4q1xvXOp6zRqqKbTq3206'
            )
        ]
    }
}

--------------------------------------------------


{
    'model': {
        'messages': [
            AIMessage(
                content='以下是客户 **cust1234** 的完整信息汇总：\n\n---\n\n### 📋 客户基本信息\n| 项目 | 内容 
|\n|------|------|\n| **姓名** | 张三 |\n| **会员等级** | 🏆 **VIP** |\n| **加入日期** | 2023-01-15 |\n\n### 📊 
历史订单\n| 项目 | 数据 |\n|------|------|\n| **总订单数** | 15 笔 |\n| **累计消费金额** | ¥25,800.00 |\n\n### 🎁 
当前可用优惠\n| 优惠活动 | 有效期至 |\n|----------|----------|\n| 🏅 老用户优惠 | 2027-01-31 |\n| 💎 会员专属折扣 |
2027-01-31 |\n\n---\n\n**总结：** 客户张三是一位 VIP 等级的老客户，自2023年1月加入至今，已完成 **15 
笔订单**，累计消费 **25,800 元**。目前有 **老用户优惠** 和 **会员专属折扣** 两项优惠可用，有效期至 
**2027年1月31日**。',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 242,
                        'prompt_tokens': 669,
                        'total_tokens': 911,
                        'completion_tokens_details': None,
                        'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 384},
                        'prompt_cache_hit_tokens': 384,
                        'prompt_cache_miss_tokens': 285
                    },
                    'model_provider': 'deepseek',
                    'model_name': 'deepseek-v4-flash',
                    'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402',
                    'id': 'bf19e458-96b3-4335-80e3-b0f6c663035c',
                    'finish_reason': 'stop',
                    'logprobs': None
                },
                id='lc_run--019f7523-357a-7221-87e7-02d3651486d7-0',
                tool_calls=[],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 669,
                    'output_tokens': 242,
                    'total_tokens': 911,
                    'input_token_details': {'cache_read': 384},
                    'output_token_details': {}
                }
            )
        ]
    }
}

--------------------------------------------------


## 3、messages输出模式

该模式中会输出流式返回的Token以及相关的元数据（如：来自哪个节点），可以用在实现类似 ChatGPT 的打字机效果场景，为聊天机器人等交互式应用提供最佳的实时体验。

In [4]:
for chunk in customer_service_agent.stream(
    {
        "messages" : [
            {"role":"user","content":"查询客户id为cust1234的完整的信息、历史订单和可用优惠"}
        ]
    },
    stream_mode="messages"
):
    # rprint(chunk)
    # print("-" * 50)
    print(chunk[0].content,end="",flush=True)

我来帮您查询客户 cust1234 的完整信息。首先，同时查询基本信息、订单历史和当前可用促销活动。{"name": "张三", "level": "VIP", "join_date": "2023-01-15"}{"total_orders": 15, "total_spent": 25800.0}{"promotions": ["老用户优惠", "会员专属折扣"], "valid_until": "2027-01-31"}以下是客户 **cust1234** 的完整信息汇总：

---

## 📋 客户基本信息
| 项目 | 内容 |
|------|------|
| **姓名** | 张三 |
| **会员等级** | 🏆 **VIP** |
| **加入日期** | 2023-01-15 |

## 🛒 历史订单
| 项目 | 数据 |
|------|------|
| **总订单数** | 15 笔 |
| **累计消费金额** | ¥25,800.00 |

## 🎁 当前可用促销活动
| 活动名称 | 有效期至 |
|---------|---------|
| ✅ **老用户优惠** | 2027-01-31 |
| ✅ **会员专属折扣** | 2027-01-31 |

---

**总结：** 张三作为 **VIP** 会员，自 2023年1月加入以来已累计消费 **25,800 元**（共15笔订单），目前可享受 **老用户优惠** 和 **会员专属折扣** 两项促销活动，有效期至2027年1月31日。

## 4、tasks输出模式

该模式会输出当前task任务开始和结束的时间，包含任务的结果和错误信息，该模式用于监控任务的生命周期。

In [6]:
for chunk in customer_service_agent.stream(
    {
        "messages" : [
            {"role":"user","content":"查询客户id为cust1234的完整的信息、历史订单和可用优惠"}
        ]
    },
    stream_mode="tasks"
):
    print(chunk)
    print("-" * 50)

{'id': '4b4bb92e-6bba-f70f-dcfa-5d91def3f531', 'name': 'model', 'input': {'messages': [HumanMessage(content='查询客户id为cust1234的完整的信息、历史订单和可用优惠', additional_kwargs={}, response_metadata={}, id='419fc8ea-bac0-4e7b-b283-e4b212b456df')]}, 'triggers': ('branch:to:model',)}
--------------------------------------------------
{'id': '4b4bb92e-6bba-f70f-dcfa-5d91def3f531', 'name': 'model', 'error': None, 'result': {'messages': [AIMessage(content='好的，我来查询客户ID为cust1234的完整信息、历史订单和当前可用优惠活动。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 121, 'prompt_tokens': 455, 'total_tokens': 576, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 384}, 'prompt_cache_hit_tokens': 384, 'prompt_cache_miss_tokens': 71}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': '96f85e29-8b10-4f68-bd05-09df5f47fd6b', 'finish_reason': 'tool_ca

## 5、debug输出模式

该模式与tasks模式类似，比task模式多输出任务步骤、时间戳、task类型（task/task_result），该模式用于调试、监控task任务的生命周期。

In [7]:
for chunk in customer_service_agent.stream(
    {
        "messages" : [
            {"role":"user","content":"查询客户id为cust1234的完整的信息、历史订单和可用优惠"}
        ]
    },
    stream_mode="debug"
):
    print(chunk)
    print("-" * 50)

{'step': 1, 'timestamp': '2026-07-18T12:14:32.347511+00:00', 'type': 'task', 'payload': {'id': '7d5c60d8-42fb-4228-c22e-a2ba277e54be', 'name': 'model', 'input': {'messages': [HumanMessage(content='查询客户id为cust1234的完整的信息、历史订单和可用优惠', additional_kwargs={}, response_metadata={}, id='60645c01-7826-4c6d-9f03-f4127309473f')]}, 'triggers': ('branch:to:model',)}}
--------------------------------------------------
{'step': 1, 'timestamp': '2026-07-18T12:14:34.071288+00:00', 'type': 'task_result', 'payload': {'id': '7d5c60d8-42fb-4228-c22e-a2ba277e54be', 'name': 'model', 'error': None, 'result': {'messages': [AIMessage(content='我来帮您查询客户ID为cust1234的完整信息、历史订单和可用优惠。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 119, 'prompt_tokens': 455, 'total_tokens': 574, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 384}, 'prompt_cache_hit_tokens': 384, 'prompt_cache_miss_tokens': 71}, 'model_provider': 'deepsee

## 6、checkpoints输出模式

该模式中，每当检查点（checkpoint）被创建时会触发输出，输出包含检查点中的状态，用于需要状态持久化、工作流恢复或分布式执行跟踪的高级场景。

In [11]:
from langgraph.checkpoint.memory import InMemorySaver

# 其他工具代码同上，保持不变
# ... ...
# 1. 创建内存检查点存储
checkpointer = InMemorySaver()

# 2. 创建Agent
customer_service_agent = create_agent(
    model=model,
    tools=[query_customer_data, check_order_history, get_current_promotions],
    checkpointer=checkpointer  # 启用检查点
)

# 3. 创建唯一的会话ID
config = {"configurable": {"thread_id": "session01"}}

# 4. 调用Agent
checkpoint_count = 0

# 使用checkpoints模式进行流式监控
for chunk in customer_service_agent.stream(
        {"messages": [{"role": "user","content": "查询客户ID为 CUST123456 的完整信息和可用优惠"}]},
        config=config,
        stream_mode="checkpoints"
):
    checkpoint_count += 1
    print(f"检查点 #{checkpoint_count}")
    print(chunk)
    print("-" * 50)

检查点 #1
{'config': {'configurable': {'checkpoint_ns': '', 'thread_id': 'session01', 'checkpoint_id': '1f160a5b-2688-644d-bfff-b364754ba359'}}, 'parent_config': None, 'values': {'messages': []}, 'metadata': {'source': 'input', 'step': -1, 'parents': {}}, 'next': ['__start__'], 'tasks': [{'id': '534e756e-f550-db45-9f9c-e4b68a2e1b1d', 'name': '__start__', 'interrupts': (), 'state': None}]}
--------------------------------------------------
检查点 #2
{'config': {'configurable': {'checkpoint_ns': '', 'thread_id': 'session01', 'checkpoint_id': '1f160a5b-268a-6f04-8000-f19b247c0c88'}}, 'parent_config': {'configurable': {'checkpoint_ns': '', 'thread_id': 'session01', 'checkpoint_id': '1f160a5b-2688-644d-bfff-b364754ba359'}}, 'values': {'messages': [HumanMessage(content='查询客户ID为 CUST123456 的完整信息和可用优惠', additional_kwargs={}, response_metadata={}, id='cc08539b-6271-402d-ad55-7f4c18c899c8')]}, 'metadata': {'source': 'loop', 'step': 0, 'parents': {}}, 'next': ['model'], 'tasks': [{'id': 'b3263de2-06bd-

## 7、custom输出模式

开发者通过`get_stream_writer`在工具或节点内部`自定义发送的数据`，用于`输出`业务逻辑相关的进度信息（如“已处理10/100条记录”）、自定义日志或指标。

In [8]:
from langchain.agents import create_agent
from langgraph.config import get_stream_writer
from langchain.tools import tool
import time

@tool
def generate_sales_report() -> str:
    """生成销售报告"""
    writer = get_stream_writer()

    writer({"type": "生成销售报告", "message": "开始生成销售报告"})

    # 模拟数据处理
    for i in range(1, 4):
        time.sleep(0.5)
        writer({"type": "生成销售报告","message": f"生成销售报告进度百分比：{i * 25}%"})

    writer({"type": "生成销售报告", "message": "报告生成完成"})

    return f"销售报告：总收入150万元，同比增长12%"


@tool
def generate_inventory_report() -> str:
    """生成库存报告"""
    writer = get_stream_writer()
    writer("开始库存分析...")
    time.sleep(0.5)
    writer("检查当前库存量...")
    time.sleep(0.5)
    writer("生成库存报告...")

    return "当前库存量为10000件，库存充足，无异常"


# 创建报告生成agent
reporting_agent = create_agent(
    model=model,
    tools=[generate_sales_report, generate_inventory_report]
)

for chunk in reporting_agent.stream(
        {"messages": [{"role": "user","content": "生成销售报告和库存报告"}]},
        stream_mode="custom"
):
    print(chunk)
    print("-" * 50)

{'type': '生成销售报告', 'message': '开始生成销售报告'}
--------------------------------------------------
开始库存分析...
--------------------------------------------------
检查当前库存量...
--------------------------------------------------
{'type': '生成销售报告', 'message': '生成销售报告进度百分比：25%'}
--------------------------------------------------
生成库存报告...
--------------------------------------------------
{'type': '生成销售报告', 'message': '生成销售报告进度百分比：50%'}
--------------------------------------------------
{'type': '生成销售报告', 'message': '生成销售报告进度百分比：75%'}
--------------------------------------------------
{'type': '生成销售报告', 'message': '报告生成完成'}
--------------------------------------------------
